# Lab 4 — Proximity Detector (LOF)

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

## Learning objectives

1. Train **Local Outlier Factor (LOF)** on **legitimate** transactions only.
2. Flag test anomalies and map LOF label **-1** → fraud prediction **1**.
3. Evaluate **precision** and **recall** on the rare fraud class.
4. Relate LOF to Day 5 DBSCAN and Day 4 KNN proximity ideas.

> **Checkpoints:** precision ≈ **0.33** · recall (fraud) = **1.00** · train legit **792** rows

**Companion script:** `../scripts/lab04_proximity_detector.py`

## Semi-supervised anomaly detection

| Step | LOF approach |
|------|----------------|
| Train | Learn density of **normal** (legit) transactions only |
| Score | Compare local density of each point to its neighbors |
| Predict | Label **-1** = outlier (low density vs neighborhood) |

**`novelty=True`** — required to call `.predict()` on new test data.

High **recall** / lower **precision** — good for a human review queue, not auto-decline.

---

## 1. Load data and split

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import OneHotEncoder, StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-06":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "credit-card" / "credit_card_transactions.csv").is_file():
            GH_ROOT = parent
            break

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_legit = X_train[y_train == 0]
print(f"train total: {len(X_train)} (fraud {int(y_train.sum())})")
print(f"train legit only: {len(X_train_legit)}")
print(f"test: {len(X_test)} (fraud {int(y_test.sum())})")

---

## 2. Preprocess and fit LOF on legit train only

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

X_train_legit_s = preprocess.fit_transform(X_train_legit)
X_test_s = preprocess.transform(X_test)

CONTAMINATION = 0.02
lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION, novelty=True)
lof.fit(X_train_legit_s)

pred = lof.predict(X_test_s)
y_pred = np.where(pred == -1, 1, 0)

print("Lab 4 — Proximity detector (LOF)")
print(f"train legit rows: {len(X_train_legit)}")
print(f"test rows: {len(X_test)}")
print(f"predicted anomalies: {int((y_pred == 1).sum())}")

---

## 3. Precision and recall (fraud)

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"precision (fraud): {precision:.4f}")
print(f"recall (fraud): {recall:.4f}")

results = pd.DataFrame({
    "metric": ["precision (fraud)", "recall (fraud)"],
    "value": [precision, recall],
})
display(results.round(4))

Recall **1.0** — both test fraud cases flagged. Precision **0.33** — 4 of 6 anomaly flags are false alarms.

---

## 4. Prediction breakdown

In [ ]:
breakdown = pd.DataFrame({
    "actual_fraud": y_test.values,
    "predicted_fraud": y_pred,
    "amount": X_test["amount"].values,
    "distance_from_home": X_test["distance_from_home"].values,
})
display(breakdown.sort_values("predicted_fraud", ascending=False).head(10).round(2))

---

## 5. Extension — vary contamination

In [ ]:
rows = []
for contam in [0.01, 0.02, 0.05]:
    model = LocalOutlierFactor(n_neighbors=20, contamination=contam, novelty=True)
    model.fit(X_train_legit_s)
    yp = np.where(model.predict(X_test_s) == -1, 1, 0)
    rows.append({
        "contamination": contam,
        "predicted_anomalies": int((yp == 1).sum()),
        "precision": precision_score(y_test, yp, zero_division=0),
        "recall": recall_score(y_test, yp, zero_division=0),
    })

display(pd.DataFrame(rows).round(4))

Higher **contamination** → more anomaly flags → recall up, precision often down.

---

## 6. LOF vs supervised (Lab 3)

In [ ]:
compare = pd.DataFrame({
    "approach": ["LOF (legit train only)", "Logistic + oversample (Lab 3)"],
    "needs_fraud_labels": ["no (train)", "yes"],
    "recall_fraud": [recall, 0.6667],
    "precision_fraud": [precision, "varies"],
})
display(compare)

---

## 7. Checkpoint summary

In [ ]:
assert len(X_train_legit) == 792
assert len(X_test) == 200
assert int((y_pred == 1).sum()) == 6
assert abs(precision - 0.3333) < 0.05
assert abs(recall - 1.0) < 0.01
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why must LOF train on legit-only data for this fraud use case?
2. When is high recall at the cost of precision acceptable?
3. How does LOF differ from DBSCAN (Day 5)?

**Previous:** [Lab 3 — Resampling](lab03_resampling_lab.ipynb)  
**Next:** [Lab 5 — Ensemble detector](lab05_ensemble_detector.ipynb)